# Skill extractor comparison

This notebook compares preserved legacy extraction with the opt-in v2 extractor on identical inputs. Reusable evaluation lives in Python modules. SkillSpan evaluates all SKILL and KNOWLEDGE BIO spans; requirement categories are evaluated separately on manually authored English/Russian fixtures. Role Radar composite scores are source metadata, never gold labels or calibrated probabilities.

Set `SKILLSPAN_PATH` to the pinned SkillSpan `data/json/test.json`. Set `ROLE_RADAR_PATH` to the local 106-pair export when available. Set `EXTRACTOR_LEXICON_PATH` only for a previously built train-only phrase lexicon. Set `SKILL_EXTRACTOR_SMOKE=1` to use a tiny authored mechanics fixture instead of making any benchmark claim.


In [ ]:
import json
import os
from pathlib import Path

from resumeanalysis.experiments.extractor_evaluation import (
    benchmark_skillspan,
    comparison_html,
    evaluate_fixtures,
    evaluate_role_radar,
    extraction_details,
    save_comparison,
)
from resumeanalysis.extraction.registry import create_extractor
from resumeanalysis.jsonio import write_json

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = str
    display = print

ROOT = Path(os.environ.get("RESUMEANALYSIS_ROOT", Path.cwd())).resolve()
if not (ROOT / "src/resumeanalysis").is_dir() and (ROOT.parent / "src/resumeanalysis").is_dir():
    ROOT = ROOT.parent
OUT = ROOT / "artifacts/experiments/skill_extractor"
OUT.mkdir(parents=True, exist_ok=True)
SKILLSPAN = Path(os.environ.get("SKILLSPAN_PATH", ROOT / "data/external/skillspan/test.json"))
ROLE_RADAR = Path(
    os.environ.get(
        "ROLE_RADAR_PATH", ROOT / "data/external/role-radar/processed/inference_pairs.jsonl"
    )
)
FIXTURES = ROOT / "tests/fixtures/skill_extraction_v2.json"
LEXICON = os.environ.get("EXTRACTOR_LEXICON_PATH") or None
SMOKE = os.environ.get("SKILL_EXTRACTOR_SMOKE") == "1"

In [ ]:
if SMOKE:
    smoke_rows = [
        {
            "tokens": ["Required", ":", "Python", ",", "Docker", "."],
            "tags_skill": ["O"] * 6,
            "tags_knowledge": ["O", "O", "B-Knowledge", "O", "B-Knowledge", "O"],
        },
        {
            "tokens": ["Communication", "is", "important", "."],
            "tags_skill": ["B-Skill", "O", "O", "O"],
            "tags_knowledge": ["O"] * 4,
        },
    ]
    SKILLSPAN = OUT / "authored_notebook_smoke.jsonl"
    SKILLSPAN.write_text("".join(json.dumps(row) + "\n" for row in smoke_rows))
    print("Authored smoke fixtures only: results do not measure SkillSpan or real-world quality.")
elif not SKILLSPAN.is_file():
    raise FileNotFoundError(
        "Set SKILLSPAN_PATH to the local pinned SkillSpan test.json, or explicitly enable SKILL_EXTRACTOR_SMOKE=1"
    )
print("Selected input:", SKILLSPAN)

## Exact-span and token metrics

Each extractor is evaluated in a fresh subprocess. The source hash must match across variants. Initialization is separate from extraction-call latency. Process peak RSS includes interpreter/import/model memory; it is not a measurement of a standalone extractor's allocations. CPU percentage uses one core as 100%.


In [ ]:
comparison = benchmark_skillspan(SKILLSPAN, ("legacy", "v2"), lexicon_path=LEXICON, isolated=True)
save_comparison(comparison, OUT / "skillspan_comparison.json")
display(HTML(comparison_html(comparison)))
for name, result in comparison["backends"].items():
    print(name, "token metrics:", result["token"])
    print(name, "false-positive examples:", result["common_false_positive_spans"][:5])
    print(name, "false-negative examples:", result["common_missed_spans"][:5])

## Side-by-side requirements and source spans

The following examples have authored expectations rather than independent recruitment labels. Alternatives remain grouped metadata, negated requirements remain separate, and titles do not create requirements. Explicit structured skill lists belong to the model-level input and remain separate evidence.


In [ ]:
extractors = {
    name: create_extractor(name, lexicon_path=LEXICON if name == "v2" else None)
    for name in ("legacy", "v2")
}
examples = [
    {
        "id": "provided-essential",
        "language": "en",
        "vacancy": True,
        "text": "Essential Skills\nFront-end: React, Next.js, TypeScript, modern CSS\nBack-end: Python/FastAPI, Node.js, Express\nAPIs: REST, GraphQL\nDatabases: PostgreSQL, MongoDB, Redis\nCloud-native deployment: Docker, Kubernetes",
    },
    {
        "id": "russian-requirements",
        "language": "ru",
        "vacancy": True,
        "text": "Обязательно знание Python и PostgreSQL. Желателен опыт работы с Docker.",
    },
    {
        "id": "alternatives-negation",
        "language": "en",
        "vacancy": True,
        "text": "Required: Python or Node.js. No Kubernetes experience required. Experience with AWS is preferred.",
    },
]
side_by_side = []
for row in examples:
    result = {"id": row["id"], "language": row["language"], "text": row["text"], "extractors": {}}
    for name, extractor in extractors.items():
        result["extractors"][name] = extraction_details(
            extractor.extract(row["text"], vacancy=True), row["text"]
        )
    side_by_side.append(result)
write_json(OUT / "side_by_side.json", side_by_side)
for row in side_by_side:
    print(row["id"], row["language"])
    for name, result in row["extractors"].items():
        print(
            name,
            {
                key: result.get(key, [])
                for key in ("required", "optional", "alternative_groups", "negated_requirements")
            },
        )

In [ ]:
fixture_results = {}
if FIXTURES.is_file():
    for name, extractor in extractors.items():
        fixture_results[name] = evaluate_fixtures(FIXTURES, extractor)
        write_json(OUT / (name + "_controlled_fixtures.json"), fixture_results[name])
        print(name, "controlled requirements by language:", fixture_results[name]["languages"])
else:
    print("Controlled fixture file is unavailable:", FIXTURES)
    print("No requirement-classification accuracy is fabricated.")

## Local Role Radar diagnostics

The external export is not committed and may be unavailable in the cloud. Every supplied source line gets a diagnostic outcome. Unknown nested fields are reported, source scores stay metadata, and no span/requirement accuracy is computed without independent annotations. Diagnostic JSON may contain extracted skill phrases and linkage metadata; keep it under ignored artifacts.


In [ ]:
if ROLE_RADAR.is_file():
    for name, extractor in extractors.items():
        diagnostics = evaluate_role_radar(ROLE_RADAR, extractor)
        write_json(OUT / (name + "_role_radar.json"), diagnostics)
        print(name, diagnostics["input_records"], diagnostics["outcome_counts"])
else:
    print("Role Radar export unavailable:", ROLE_RADAR)
    print("Run locally after setting ROLE_RADAR_PATH. No access or accuracy is assumed.")
print("Saved reports:", OUT)

## Equivalent CLI

```bash
resumeanalysis extractors compare --input "$SKILLSPAN_PATH" \
  --extractors legacy v2 --output artifacts/experiments/skill_extractor/skillspan_comparison.json
resumeanalysis extractors fixtures --extractor v2 \
  --input tests/fixtures/skill_extraction_v2.json \
  --output artifacts/experiments/skill_extractor/v2_controlled_fixtures.json
resumeanalysis extractors role-radar --extractor v2 \
  --input data/external/role-radar/processed/inference_pairs.jsonl \
  --output artifacts/experiments/skill_extractor/v2_role_radar.json
```

An optional train-only lexicon must be built without evaluation text or labels. Pass its path explicitly with `--lexicon-path` and preserve its source/split/license provenance. Increased extraction recall alone does not establish improved matching: use versioned, appropriately retrained MLP checkpoints for downstream comparisons.
